# 07 — Export to ONNX

Convert the trained YOLOv11 checkpoint (`weights/best.pt`) to `weights/best.onnx` for deployment in ONNX Runtime, OpenCV DNN, or other runtimes.

In [4]:
# Install dependencies
%pip install -q "ultralytics==8.3.*" "onnx>=1.12.0" onnxruntime
print("nb07 dependencies ready")

Note: you may need to restart the kernel to use updated packages.
nb07 dependencies ready


In [5]:
import shutil
from pathlib import Path

from ultralytics import YOLO

In [6]:
# ── Config ─────────────────────────────────────────────────────────────────────────
WEIGHTS = Path('../weights/best.pt').resolve()
ONNX_OUT = WEIGHTS.with_suffix('.onnx')

IMGSZ   = 640
OPSET   = 12
DYNAMIC = True    # dynamic batch/HW axes
SIMPLIFY = True   # run onnx-simplifier
HALF    = False   # fp16 export (GPU only)

assert WEIGHTS.exists(), f'Weights not found: {WEIGHTS} \u2014 run notebook 04 first'
print(f'Source  : {WEIGHTS}')
print(f'Target  : {ONNX_OUT}')

Source  : /Users/th/github/ai_cv_project/weights/best.pt
Target  : /Users/th/github/ai_cv_project/weights/best.onnx


In [7]:
# ── Export ─────────────────────────────────────────────────────────────────────────
model = YOLO(str(WEIGHTS))
exported = model.export(
    format='onnx',
    imgsz=IMGSZ,
    opset=OPSET,
    dynamic=DYNAMIC,
    simplify=SIMPLIFY,
    half=HALF,
)
exported = Path(exported)

# Ultralytics writes <stem>.onnx next to the .pt — move it to ONNX_OUT if needed
if exported.resolve() != ONNX_OUT:
    shutil.move(str(exported), ONNX_OUT)

print(f'Exported: {ONNX_OUT}  ({ONNX_OUT.stat().st_size/1e6:.1f} MB)')

Ultralytics 8.3.253 🚀 Python-3.14.0 torch-2.11.0 CPU (Apple M1 Pro)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/Users/th/github/ai_cv_project/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.2 MB)

ONNX: starting export with onnx 1.21.0 opset 12...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 2.8s, saved as '/Users/th/github/ai_cv_project/weights/best.onnx' (10.2 MB)

Export complete (3.1s)
Results saved to /Users/th/github/ai_cv_project/weights
Predict:         yolo predict task=detect model=/Users/th/github/ai_cv_project/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=/Users/th/github/ai_cv_project/weights/best.onnx imgsz=640 data=/Users/th/github/ai_cv_project/data/dataset/data.yaml  
Visualize:       https://netron.app
Exported: /Users/th/github/ai_cv_project/weights/best.onnx  (10.7 MB)


In [8]:
# ── Verify ─────────────────────────────────────────────────────────────────────────
import numpy as np
import onnx
import onnxruntime as ort

onnx_model = onnx.load(str(ONNX_OUT))
onnx.checker.check_model(onnx_model)

sess = ort.InferenceSession(str(ONNX_OUT), providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
out = sess.get_outputs()[0]
print(f'Input   : {inp.name}  shape={inp.shape}  dtype={inp.type}')
print(f'Output  : {out.name}  shape={out.shape}  dtype={out.type}')

dummy = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)
pred = sess.run(None, {inp.name: dummy})[0]
print(f'Dummy forward pass OK \u2014 output shape: {pred.shape}')

Input   : images  shape=['batch', 3, 'height', 'width']  dtype=tensor(float)
Output  : output0  shape=['batch', 8, 'anchors']  dtype=tensor(float)
Dummy forward pass OK — output shape: (1, 8, 8400)
